# HGRIA - Hand Gesture Recognition for Interactive Applications
## Launch Notebook

This notebook sets up and launches the HGRIA server with webcam support.

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Mount Google Drive for logging (optional)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Start ngrok tunnel
!pip install -q pyngrok
from pyngrok import ngrok

# Get public URL
tunnel = ngrok.connect(5000, 'http')
public_url = tunnel.public_url.replace('http://', 'https://')
print(f'Public URL: {public_url}')

In [ ]:
# Start webcam bridge (JavaScript)
%%javascript

const SERVER_URL = 'PLACEHOLDER_URL';
const TARGET_FPS = 30;
const INTERVAL_MS = 1000 / TARGET_FPS;

async function startWebcam() {
    try {
        const stream = await navigator.mediaDevices.getUserMedia({ 
            video: { width: 640, height: 480, facingMode: 'user' } 
        });
        const video = document.createElement('video');
        video.srcObject = stream;
        video.play();

        const canvas = document.createElement('canvas');
        canvas.width = 640;
        canvas.height = 480;
        const ctx = canvas.getContext('2d');

        setInterval(async () => {
            ctx.drawImage(video, 0, 0, 640, 480);
            const b64 = canvas.toDataURL('image/jpeg', 0.7);
            try {
                await fetch(`${SERVER_URL}/api/frame`, {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ image: b64 }),
                });
            } catch (e) {
                console.warn('Frame send failed:', e);
            }
        }, INTERVAL_MS);

        console.log('Webcam bridge started');
    } catch (e) {
        console.error('Webcam access denied:', e);
        alert('Please allow webcam access to use gesture recognition.');
    }
}

startWebcam();

In [ ]:
# Start the HGRIA server
import sys
sys.path.insert(0, '/content/HGRIA')

from backend.main import SystemOrchestrator

# Update ngrok URL placeholder
print(f'Open this URL in your browser: {public_url}')

# Start the orchestrator
orchestrator = SystemOrchestrator('/content/HGRIA/config/config.json')
orchestrator.start()

## Instructions

1. Open the public URL in your browser
2. Allow webcam access when prompted
3. Make hand gestures in front of the camera
4. The game will respond to your gestures